# Isoflurane NVU Diffusion — Quick Start

This notebook walks through the key model components:
1. The D_BBB(c) Hill function and its calibration
2. Running the v0.2 solver
3. Reproducing the key sensitivity result (K_bt invariance)
4. Generating the 2D visualization

In [ ]:
import sys
sys.path.insert(0, '../src')
import numpy as np
import matplotlib.pyplot as plt
from isoflurane_figures import run_sim, D_BBB_0, D_BBB_max, c_MAC, K_bt_base

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. The D_BBB(c) Hill Function

The key innovation of v0.2: BBB diffusivity opens as a sigmoidal function of local isoflurane concentration.

In [ ]:
c_vessel = c_MAC * K_bt_base
K_half   = c_vessel / (3.5 ** 0.5)
n_hill   = 2.0

c_range = np.linspace(0, 2.5 * c_vessel, 300)
D_BBB_c = D_BBB_0 + (D_BBB_max - D_BBB_0) * c_range**n_hill / (K_half**n_hill + c_range**n_hill)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(c_range / c_vessel, D_BBB_c / D_BBB_0, color='darkorange', lw=2.5)
ax.axvline(1.0, color='purple', ls='--', label='1 MAC')
ax.scatter([0, 0.87, 2.61], [1.0, 7.83, 103.8],
           color='black', zorder=5, label='Tetrault 2008 data')
ax.set(xlabel='c / c_vessel', ylabel='D_BBB fold over baseline',
       title='Concentration-dependent BBB diffusivity')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()

## 2. Run the solver and plot radial profiles

In [ ]:
r, snaps, cv, idx_n = run_sim(t_saves=[10, 30, 60, 120, 300])
r_um = r * 1e6

fig, ax = plt.subplots(figsize=(9, 5))
colors = plt.cm.plasma(np.linspace(0.15, 0.9, 5))
for i, ts in enumerate([10, 30, 60, 120, 300]):
    if ts in snaps:
        ax.plot(r_um, snaps[ts]/cv, color=colors[i], lw=2, label=f't={ts}s')

ax.axvspan(5, 6, alpha=0.2, color='orange', label='BBB')
ax.axvspan(6, 15, alpha=0.1, color='green', label='ECS')
ax.set(xlabel='r (μm)', ylabel='c / c_vessel',
       title='Radial concentration profiles — D_BBB(c) model')
ax.legend(fontsize=9); ax.grid(alpha=0.25)
plt.tight_layout()

## 3. The K_bt invariance result

Varying K_bt (brain:blood partition) produces identical normalized concentration profiles.
This is the key finding: partition coefficient sets absolute dose, not delivery kinetics.

In [ ]:
t_saves_s = list(range(5, 125, 5))
K_bt_vals = [0.8, 1.1, 1.4, 1.8, 2.2]
colors_k  = plt.cm.Greens(np.linspace(0.4, 0.9, len(K_bt_vals)))

fig, ax = plt.subplots(figsize=(8, 5))
for k_bt, col in zip(K_bt_vals, colors_k):
    r_s, snaps_s, cv_s, idx_s = run_sim(K_bt_p=k_bt, t_end=120, t_saves=t_saves_s)
    c_n = [snaps_s.get(ts, np.zeros(len(r_s)))[idx_s] / cv_s * 100 for ts in t_saves_s]
    ax.plot(t_saves_s, c_n, lw=2, color=col, label=f'K_bt={k_bt}')

ax.axhline(90, color='gray', ls=':', label='90% threshold')
ax.set(xlabel='Time (s)', ylabel='c_neuron (% of c_vessel)',
       title='K_bt invariance — all curves overlap (normalized kinetics are identical)')
ax.legend(fontsize=9); ax.grid(alpha=0.25)
plt.tight_layout()
print('All t90 values:')
# They should all be ~11.7s